# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [16]:
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [ ]:
#built func loads of times with alot of erroer answer is was very helped i did not acheive scraping glory need more practice

def scrape_books(min_rating, max_price):

    base_url = "https://books.toscrape.com/"
    url = base_url

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    books_data = []

    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    while url:

        print("Scraping:", url)

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        # Stop if the page cannot be accessed
        if response.status_code != 200:
            print(
                f"Could not access page: "
                f"{response.status_code}"
            )
            break

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        books = soup.select(
            "article.product_pod"
        )

        for book in books:

            # -------------------------
            # TITLE
            # -------------------------
            title = book.h3.a["title"]

            # -------------------------
            # PRICE
            # -------------------------
            price_text = book.select_one(
                ".price_color"
            ).get_text(strip=True)

            price_match = re.search(
                r"\d+\.\d+",
                price_text
            )

            if not price_match:
                continue

            price = float(
                price_match.group()
            )

            # -------------------------
            # RATING
            # -------------------------
            rating_classes = book.select_one(
                ".star-rating"
            )["class"]

            rating_word = next(
                word
                for word in rating_classes
                if word in rating_map
            )

            rating = rating_map[rating_word]

            # -------------------------
            # FILTER
            # -------------------------
            if rating < min_rating:
                continue

            if price > max_price:
                continue

            # -------------------------
            # AVAILABILITY
            # -------------------------
            availability = book.select_one(
                ".availability"
            ).get_text(
                " ",
                strip=True
            )

            # -------------------------
            # BOOK URL
            # -------------------------
            book_link = book.h3.a["href"]

            book_url = urljoin(
                url,
                book_link
            )

            # -------------------------
            # DETAIL PAGE
            # -------------------------
            book_response = requests.get(
                book_url,
                headers=headers,
                timeout=10
            )

            if book_response.status_code != 200:
                print(
                    f"Skipping: {title} "
                    f"({book_response.status_code})"
                )
                continue

            book_soup = BeautifulSoup(
                book_response.text,
                "html.parser"
            )

            # -------------------------
            # UPC
            # -------------------------
            upc = None

            for row in book_soup.select(
                "table.table-striped tr"
            ):

                cells = row.find_all("td")

                if len(cells) == 2:

                    label = cells[0].get_text(
                        strip=True
                    )

                    if label == "UPC":
                        upc = cells[1].get_text(
                            strip=True
                        )
                        break

            # -------------------------
            # GENRE
            # -------------------------
            breadcrumbs = book_soup.select(
                "ul.breadcrumb li a"
            )

            if breadcrumbs:
                genre = breadcrumbs[-1].get_text(
                    strip=True
                )
            else:
                genre = None

            # -------------------------
            # DESCRIPTION
            # -------------------------
            description_header = book_soup.find(
                "div",
                id="product_description"
            )

            if description_header:

                description_tag = (
                    description_header
                    .find_next_sibling("p")
                )

                if description_tag:
                    description = (
                        description_tag
                        .get_text(strip=True)
                    )
                else:
                    description = None

            else:
                description = None

            # -------------------------
            # SAVE
            # -------------------------
            books_data.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": rating,
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })

        # -------------------------
        # NEXT PAGE
        # -------------------------
        next_button = soup.select_one(
            "li.next a"
        )

        if next_button:

            next_link = next_button["href"]

            url = urljoin(
                url,
                next_link
            )

        else:
            url = None

    return pd.DataFrame(
        books_data
    )

In [18]:
df = scrape_books(min_rating=4, max_price=20)

df

Scraping: https://books.toscrape.com/
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html
Scraping: https://books.toscrape.com/catalogue/page-4.html
Scraping: https://books.toscrape.com/catalogue/page-5.html
Scraping: https://books.toscrape.com/catalogue/page-6.html
Scraping: https://books.toscrape.com/catalogue/page-7.html
Scraping: https://books.toscrape.com/catalogue/page-8.html
Scraping: https://books.toscrape.com/catalogue/page-9.html
Scraping: https://books.toscrape.com/catalogue/page-10.html
Scraping: https://books.toscrape.com/catalogue/page-11.html
Scraping: https://books.toscrape.com/catalogue/page-12.html
Scraping: https://books.toscrape.com/catalogue/page-13.html
Scraping: https://books.toscrape.com/catalogue/page-14.html
Scraping: https://books.toscrape.com/catalogue/page-15.html
Scraping: https://books.toscrape.com/catalogue/page-16.html
Scraping: https://books.toscrape.com/catalogue/page-17.html
Scraping: 

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,None,Set Me Free,17.46,5,Young Adult,In stock,Aaron Ledbetterâs future had been planned ou...
1,None,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock,"In The Four Agreements, don Miguel Ruiz reveal..."
2,None,Sophie's World,15.94,5,Philosophy,In stock,A page-turning novel that is also an explorati...
3,None,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock,"More than thirty-five years ago, when the weat..."
4,None,This One Summer,19.49,4,Sequential Art,In stock,"Every summer, Rose goes with her mom and dad t..."
...,...,...,...,...,...,...,...
70,None,The Zombie Room,19.69,5,Default,In stock,An unlikely bond is forged between three men f...
71,None,The Silent Wife,12.34,5,Fiction,In stock,A chilling psychological thriller about a marr...
72,None,The Girl You Lost,12.29,5,Mystery,In stock,Eighteen years ago your baby daughter was snat...
73,None,The Edge of Reason (Bridget Jones #2),19.18,4,Womens Fiction,In stock,Monday 27 Januaryâ7:15 a.m. Hurrah! The wild...
